In [1]:
from dotenv import load_dotenv
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

load_dotenv()


True

In [3]:
from langchain_openai import ChatOpenAI
import os

llm= ChatOpenAI(
    base_url=os.getenv("OCI_LLM_BASE_URL"),
    api_key=os.getenv("OCI_LLM_API_KEY"),
    model=os.getenv("OCI_MODEL_QWEN"),
    temperature=0.2
)

class State(TypedDict):
    messages: Annotated[list,add_messages]


def chatbot(state:State) -> State:
    return {"messages":[llm.invoke(state["messages"])]}

builder = StateGraph(State)
builder.add_node("chatbot_node", chatbot)

builder.add_edge(START, "chatbot_node")
builder.add_edge("chatbot_node", END)

graph = builder.compile()

In [4]:
message ={"role": "user", "content": "Who walked on the Antarctica for the fist time? Print only the name"}
response=graph.invoke({"messages":[message]})
response

{'messages': [HumanMessage(content='Who walked on the Antarctica for the fist time? Print only the name', additional_kwargs={}, response_metadata={}, id='b46415f6-7245-4eb6-845d-8c201272d833'),
  AIMessage(content='Roald Amundsen', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 6, 'prompt_tokens': 22, 'total_tokens': 28, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 1}}, 'model_provider': 'openai', 'model_name': 'Qwen3-30B-A3B-Instruct-2507-IQ4_NL.gguf', 'system_fingerprint': 'b12282-aa75ff323', 'id': 'chatcmpl-Qk8djb4RKqFi5pdIhxeUkm0Y2GRUpJnC', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1f8c6-d65d-7d72-8d52-018390a9d649-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 22, 'output_tokens': 6, 'total_tokens': 28, 'input_token_details': {'cache_read': 1}, 'output_token_details': {}})]}

In [5]:
state = None
while True:
    in_message = input("You: ")
    if in_message.lower() in {"quit","exit"}:
        break
    if state is None:
        state: State = {
            "messages": [{"role": "user", "content": in_message}]
        }
    else:
        state["messages"].append({"role": "user", "content": in_message})

    state = graph.invoke(state)
    print("Bot:", state["messages"][-1].content)

Bot: Antarctica is located at the southernmost part of Earth, centered around the South Pole. It is the fifth largest continent and is almost entirely covered by ice. Antarctica lies in the Southern Hemisphere, surrounded by the Southern Ocean (also known as the Antarctic Ocean), which connects to the Pacific, Atlantic, and Indian Oceans.

It's situated roughly between 180°W and 180°E longitude and from about 60°S latitude southward to the South Pole. Because of its extreme climate and remote location, it is one of the most isolated places on Earth.

No country owns Antarctica—its landmass is governed by an international agreement called the **Antarctic Treaty System**, which preserves the continent for scientific research and bans military activity and mineral mining.


KeyboardInterrupt: Interrupted by user